# Building a Neo4j Knowledge Graph with nano-graphrag

This notebook uses **nano-graphrag** with **Ollama** (local LLM + embeddings) and **Neo4j** as graph storage to build a knowledge graph from the document in `inputs/`.

In [17]:
import os
import json
import logging
import textwrap
from neo4j import GraphDatabase
import ollama

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("neo4j-graphbuilder")

## Configuration

Set your **Ollama model name** and **Neo4j credentials** below. Make sure:
- Neo4j is running (e.g. via Docker: `docker run -d --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/testpassword neo4j:5`)
- Ollama is running with the model pulled

In [18]:
! ollama list

NAME                                                 ID              SIZE      MODIFIED    
hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest    042cf58aa32f    807 MB    10 days ago    
llama3:8b                                            365c0bd3c000    4.7 GB    10 days ago    
ORANSight_Qwen_3B_Instruct:latest                    d578a3f030a6    6.2 GB    10 days ago    
ORANSight_Qwen_1.5B_Instruct:latest                  07bc24db663e    3.1 GB    10 days ago    
phi:latest                                           e2fd6321a5fe    1.6 GB    10 days ago    
tinyllama:latest                                     2644915ede35    637 MB    10 days ago    
hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest    98c4eb4a3287    68 MB     2 weeks ago    
mxbai-embed-large:latest                             468836162de7    669 MB    6 weeks ago    


In [19]:
# ── Ollama model ──
LLM_MODEL = "llama3:8b"  # Change to your preferred Ollama model

# ── Neo4j connection ──
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "testpassword"

# ── Chunking (larger chunks = fewer LLM calls) ──
CHUNK_SIZE = 6000       # characters per chunk
CHUNK_OVERLAP = 300     # overlap between chunks

## Load Document and Chunk It

Read the text file from `inputs/` and split into overlapping chunks for the LLM context window.

In [20]:
INPUT_FILE = os.path.join("inputs", "2408.13296v3.txt")

with open(INPUT_FILE, encoding="utf-8-sig") as f:
    document_text = f.read()

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(document_text)
print(f"Loaded: {os.path.basename(INPUT_FILE)}")
print(f"Total characters: {len(document_text):,}")
print(f"Number of chunks: {len(chunks)}")

Loaded: 2408.13296v3.txt
Total characters: 292,045
Number of chunks: 52


## Entity & Relationship Extraction with Ollama

For each chunk, prompt the LLM to extract entities (with type and description) and relationships as structured JSON.

In [21]:
EXTRACTION_PROMPT = textwrap.dedent("""\
You are an expert at extracting entities and relationships from text for building a knowledge graph.

Given the following text chunk, extract:
1. **Entities**: each with a "name", "type" (e.g. CONCEPT, TECHNOLOGY, MODEL, PERSON, ORGANIZATION, METHOD, DATASET, METRIC), and a short "description".
2. **Relationships**: each with a "source" entity name, "target" entity name, "relation" label (e.g. USES, PART_OF, DEVELOPED_BY, BASED_ON, EVALUATES, IMPROVES), and a short "description".

Return ONLY valid JSON in this exact format:
{
  "entities": [
    {"name": "...", "type": "...", "description": "..."}
  ],
  "relationships": [
    {"source": "...", "target": "...", "relation": "...", "description": "..."}
  ]
}

TEXT:
""")

def extract_from_chunk(chunk_text, chunk_index, max_retries=2):
    """Use Ollama to extract entities and relationships from a text chunk."""
    prompt = EXTRACTION_PROMPT + chunk_text
    for attempt in range(max_retries + 1):
        try:
            response = ollama.chat(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                format="json",  # Force Ollama to return valid JSON
                options={"temperature": 0.0},
            )
            content = response["message"]["content"]
            data = json.loads(content.strip())
            # Validate expected structure
            if "entities" not in data:
                data["entities"] = []
            if "relationships" not in data:
                data["relationships"] = []
            n_ent = len(data["entities"])
            n_rel = len(data["relationships"])
            log.info(f"Chunk {chunk_index}: {n_ent} entities, {n_rel} relationships")
            return data
        except (json.JSONDecodeError, KeyError, IndexError) as e:
            if attempt < max_retries:
                log.warning(f"Chunk {chunk_index}: Retry {attempt+1} — {e}")
            else:
                log.warning(f"Chunk {chunk_index}: Failed after {max_retries+1} attempts — {e}")
                return {"entities": [], "relationships": []}

print("Extraction function defined.")

Extraction function defined.


## Run Extraction on All Chunks

Process each chunk through the LLM and collect all extracted entities and relationships.

In [22]:
import time

# Resume-friendly: only reset if not already populated
if "all_entities" not in dir() or not all_entities:
    all_entities = {}
    all_relationships = []
    start_chunk = 0
else:
    start_chunk = i + 1  # resume from where we left off
    print(f"Resuming from chunk {start_chunk} (already have {len(all_entities)} entities, {len(all_relationships)} rels)")

failed_chunks = []

for i in range(start_chunk, len(chunks)):
    chunk = chunks[i]
    t0 = time.time()
    result = extract_from_chunk(chunk, i)
    elapsed = time.time() - t0

    n_new_ent = 0
    for ent in result.get("entities", []):
        name = ent.get("name", "").strip().upper()
        if name:
            all_entities[name] = {
                "type": ent.get("type", "CONCEPT").strip().upper(),
                "description": ent.get("description", ""),
            }
            n_new_ent += 1

    n_new_rel = 0
    for rel in result.get("relationships", []):
        src = rel.get("source", "").strip().upper()
        tgt = rel.get("target", "").strip().upper()
        if src and tgt:
            all_relationships.append({
                "source": src,
                "target": tgt,
                "relation": rel.get("relation", "RELATED_TO").strip().upper(),
                "description": rel.get("description", ""),
            })
            n_new_rel += 1

    if n_new_ent == 0 and n_new_rel == 0:
        failed_chunks.append(i)

    print(f"[{i+1}/{len(chunks)}] +{n_new_ent} ent, +{n_new_rel} rel ({elapsed:.1f}s) | Total: {len(all_entities)} entities, {len(all_relationships)} rels")

print(f"\n{'='*60}")
print(f"Extraction complete!")
print(f"Unique entities: {len(all_entities)}")
print(f"Relationships:   {len(all_relationships)}")
if failed_chunks:
    print(f"Failed/empty chunks: {failed_chunks}")

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:neo4j-graphbuilder:Chunk 0: 10 entities, 7 relationships


[1/52] +10 ent, +7 rel (22.0s) | Total: 10 entities, 7 rels


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:neo4j-graphbuilder:Chunk 1: 3 entities, 2 relationships


[2/52] +3 ent, +2 rel (7.1s) | Total: 12 entities, 9 rels


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:neo4j-graphbuilder:Chunk 2: 3 entities, 2 relationships


...

[50/52] +9 ent, +8 rel (21.8s) | Total: 353 entities, 334 rels


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:neo4j-graphbuilder:Chunk 50: 7 entities, 6 relationships


[51/52] +7 ent, +6 rel (17.8s) | Total: 357 entities, 340 rels


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:neo4j-graphbuilder:Chunk 51: 3 entities, 2 relationships


[52/52] +3 ent, +2 rel (5.4s) | Total: 358 entities, 342 rels

Extraction complete!
Unique entities: 358
Relationships:   342


## Connect to Neo4j and Build the Graph

Create a uniqueness constraint, then insert all entities as nodes and relationships as edges.

In [31]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Verify connectivity
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

# ── Create constraint for unique entity names ──
with driver.session() as session:
    session.run(
        "CREATE CONSTRAINT entity_name_unique IF NOT EXISTS "
        "FOR (e:Entity) REQUIRE e.name IS UNIQUE"
    )
    print("Uniqueness constraint ensured.")

INFO:neo4j.notifications:Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT entity_name_unique IF NOT EXISTS FOR (e:Entity) REQUIRE (e.name) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT entity_name_unique FOR (e:Entity) REQUIRE (e.name) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT entity_name_unique IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE'


Connected to Neo4j successfully!
Uniqueness constraint ensured.


In [32]:
# ── Insert entities as nodes ──
with driver.session() as session:
    for name, props in all_entities.items():
        session.run(
            "MERGE (e:Entity {name: $name}) "
            "SET e.type = $type, e.description = $description",
            name=name, type=props["type"], description=props["description"],
        )
    print(f"Inserted {len(all_entities)} entity nodes.")

# ── Insert relationships as edges ──
with driver.session() as session:
    for rel in all_relationships:
        session.run(
            "MATCH (a:Entity {name: $source}) "
            "MATCH (b:Entity {name: $target}) "
            "MERGE (a)-[r:RELATES_TO {relation: $relation}]->(b) "
            "SET r.description = $description",
            source=rel["source"],
            target=rel["target"],
            relation=rel["relation"],
            description=rel["description"],
        )
    print(f"Inserted {len(all_relationships)} relationships.")

Inserted 358 entity nodes.
Inserted 342 relationships.


## Verify the Graph

Check node and relationship counts, and preview some data.

In [33]:
with driver.session() as session:
    # Count nodes and relationships
    node_count = session.run("MATCH (n:Entity) RETURN count(n) AS cnt").single()["cnt"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS cnt").single()["cnt"]
    print(f"Graph has {node_count} nodes and {rel_count} relationships.\n")

    # Show entity type distribution
    print("Entity types:")
    result = session.run(
        "MATCH (n:Entity) RETURN n.type AS type, count(*) AS cnt ORDER BY cnt DESC"
    )
    for record in result:
        print(f"  {record['type']:20s} {record['cnt']}")

    # Show sample relationships
    print("\nSample relationships (first 10):")
    result = session.run(
        "MATCH (a:Entity)-[r]->(b:Entity) "
        "RETURN a.name AS src, r.relation AS rel, b.name AS tgt LIMIT 10"
    )
    for record in result:
        print(f"  {record['src']} --[{record['rel']}]--> {record['tgt']}")

Graph has 358 nodes and 222 relationships.

Entity types:
  TECHNOLOGY           105
  MODEL                67
  CONCEPT              38
  METHOD               36
  DATASET              34
  ORGANIZATION         28
  METRIC               13
  TOOL                 13
  LIBRARY              8
  SERVICE              3
  FINE_TUNING_TECHNIQUE 3
  FRAMEWORK            2
  BASE_MODEL           2
  PYTHON_LIBRARY       2
  TASK                 1
  MODEL ARCHITECTURE   1
  PLATFORM             1
  PROJECT              1

Sample relationships (first 10):
  LARGE LANGUAGE MODELS (LLMS) --[USES]--> TRANSFORMER
  GPT-3 --[DEVELOPED_BY]--> REINFORCEMENT LEARNING FROM HUMAN FEEDBACK (RLHF)
  N-GRAM MODELS --[PART_OF]--> LARGE LANGUAGE MODELS (LLMS)
  PALM --[IS_A]--> LARGE LANGUAGE MODELS (LLMS)
  LLAMA --[IS_A]--> LARGE LANGUAGE MODELS (LLMS)
  LARGE LANGUAGE MODELS (LLMS) --[USES]--> OPENAI'S GPT SERIES
  LARGE LANGUAGE MODELS (LLMS) --[BASED_ON]--> TABLE 1.1
  FINE-TUNING --[ADJUSTS_WEIGHTS]--> L

## Query the Knowledge Graph

Use Cypher to retrieve context from Neo4j, then pass it to Ollama to answer questions about the document.

In [34]:
def graph_rag_query(question, driver, model=LLM_MODEL, top_k=20):
    """
    1. Search Neo4j for entities matching keywords in the question.
    2. Retrieve their neighbourhood (connected nodes + relationships).
    3. Build a context string and send it to Ollama for answer generation.
    """
    # ── Step 1: Find relevant entities via full-text-like keyword matching ──
    with driver.session() as session:
        # Use case-insensitive CONTAINS on entity names/descriptions
        keywords = [w for w in question.split() if len(w) > 3]
        where_clause = " OR ".join(
            [f"toLower(n.name) CONTAINS toLower($kw{i}) OR toLower(n.description) CONTAINS toLower($kw{i})"
             for i in range(len(keywords))]
        )
        params = {f"kw{i}": kw for i, kw in enumerate(keywords)}

        if not where_clause:
            where_clause = "TRUE"

        # Get matching nodes and their 1-hop neighbourhood
        cypher = (
            f"MATCH (n:Entity) WHERE {where_clause} "
            "WITH n LIMIT $top_k "
            "OPTIONAL MATCH (n)-[r]-(m:Entity) "
            "RETURN n.name AS entity, n.type AS type, n.description AS desc, "
            "       r.relation AS rel, m.name AS connected_to "
            "LIMIT 200"
        )
        params["top_k"] = top_k
        result = session.run(cypher, **params)

        context_lines = []
        for record in result:
            line = f"Entity: {record['entity']} (type: {record['type']})"
            if record["desc"]:
                line += f" — {record['desc']}"
            if record["connected_to"]:
                line += f" | Relation: {record['rel']} → {record['connected_to']}"
            context_lines.append(line)

    if not context_lines:
        context_str = "No relevant entities found in the knowledge graph."
    else:
        context_str = "\n".join(context_lines)

    # ── Step 2: Generate answer with Ollama ──
    prompt = (
        f"Based on the following knowledge graph context, answer the question.\n\n"
        f"CONTEXT:\n{context_str}\n\n"
        f"QUESTION: {question}\n\n"
        f"Provide a detailed, well-structured answer based only on the context above."
    )

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3},
    )
    return response["message"]["content"]

print("Query function defined.")

Query function defined.


In [35]:
# ── Example query ──
question = "What are the main techniques used in Large Language Models?"
answer = graph_rag_query(question, driver)

print(f"Q: {question}\n")
print(f"A: {answer}")

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Q: What are the main techniques used in Large Language Models?

A: Based on the provided knowledge graph context, the main techniques used in Large Language Models (LLMs) can be summarized as follows:

1. **Fine-Tuning**: This technique trains LLMs on smaller, domain-specific datasets for specific tasks. Fine-tuning is supported by various methods, including Tensorflow Privacy and Federated Learning.
2. **Parameter-Efficient Fine-Tuning (PEFT)**: PEFT is a NLP technique that adapts pre-trained language models to various applications with remarkable efficiency. It uses Adapter Layers to efficiently update the model's parameters.
3. **Half Fine-Tuning (HFT)**: HFT is a technique designed to balance the retention of foundational knowledge with the acquisition of new skills in LLMs. It evaluates performance on LLAMA 2-7B and applies to large language models.

These techniques are used to customize, adapt, and fine-tune LLMs for various applications, including NLP tasks such as summarizatio

## Cleanup (Optional)

Close the Neo4j driver when done. To delete all graph data, uncomment the `DELETE` line.

In [28]:
# Uncomment the following to delete all nodes and relationships:
# with driver.session() as session:
#     session.run("MATCH (n) DETACH DELETE n")
#     print("All graph data deleted.")

driver.close()
print("Neo4j driver closed.")

Neo4j driver closed.
